In [27]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Load the IndicTrans2 model and tokenizer
model_name = "ai4bharat/indictrans2-en-indic-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, trust_remote_code=True)

# Ensure the model runs on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define the translation function
def translate_english_to_bengali(text, max_length=512):
    # Correct language tags for IndicTrans2
    # English: eng_Latn, Bengali: ben_Beng
    text_with_tags = f"eng_Latn ben_Beng {text}"

    # Tokenize the input text
    inputs = tokenizer(text_with_tags, return_tensors="pt", padding=True, truncation=True, max_length=max_length)
    inputs = {key: val.to(device) for key, val in inputs.items()}

    # Generate translation
    translated_tokens = model.generate(
        **inputs,
        max_length=max_length,
        num_beams=5,
        length_penalty=1.0,
        early_stopping=True
    )

    # Decode the translated tokens
    translated_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)
    return translated_text

# Example usage
english_text = "Hello, how are you today?"
bengali_translation = translate_english_to_bengali(english_text)
print(f"English: {english_text}")
print(f"Bengali: {bengali_translation}")

# Additional examples
examples = [
    "Good morning!",
    "Thank you for your help.",
    "I am learning Bengali.",
    "The weather is nice today."
]

print("\nMore translation examples:")
for text in examples:
    translation = translate_english_to_bengali(text)
    print(f"EN: {text}")
    print(f"BN: {translation}")
    print("-" * 50)

English: Hello, how are you today?
Bengali: आससालामुय़ालाइकुम, आज केमन आछ?

More translation examples:
EN: Good morning!
BN: शुभ सकाल ।
--------------------------------------------------
EN: Thank you for your help.
BN: आपनार साहाय्येर जन्य धन्यबाद ।
--------------------------------------------------
EN: I am learning Bengali.
BN: आमि बांला शिखछि ।
--------------------------------------------------
EN: The weather is nice today.
BN: आज आबहाओय़ा भालो ।
--------------------------------------------------
